### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="gallstone_disease",
    dataset_year="2023",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.1097/md.0000000000037258",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/1150/gallstone-1.zip && unzip gallstone-1.zip && rm gallstone-1.zip && unzip dataset-uci.zip && rm dataset-uci.zip && mkdir -p local-data-warehouse/gallstone_disease && mv dataset-uci.xlsx local-data-warehouse/gallstone_disease/
""",
    # References
    academic_reference_bibtex=r"""@article{esen2024early,
  title={Early prediction of gallstone disease with a machine learning-based method from bioimpedance and laboratory data},
  author={Esen, {\.I}rfan and Arslan, Hilal and Esen, Selin Akt{\"u}rk and G{\"u}l{\c{s}}en, Mervenur and K{\"u}ltekin, Nimet and {\"O}zdemir, O{\u{g}}uzhan},
  journal={Medicine},
  volume={103},
  number={8},
  pages={e37258},
  year={2024},
  publisher={LWW}
}
""",
    academic_reference_bibtex_key="esen2024early",
    licence="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We use the data without any further curation.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Gallstone Status",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Gallstone Status",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel(dataset_mold.path / "dataset-uci.xlsx", sheet_name="dataset", engine="calamine")

as_cat_type = [
    "Gallstone Status", "Coronary Artery Disease (CAD)", "Hypothyroidism",
    "Gender", "Comorbidity", "Hyperlipidemia", "Diabetes Mellitus (DM)",
]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 319
Columns: 39
Use sampling: False (sample size: 319)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Lean Mass (LM) (%)', 'Obesity (%)', 'Glomerular Filtration Rate (GFR)', 'Body Protein Content (Protein) (%)', 'Weight', 'Vitamin D', 'Total Body Fat Ratio (TBFR) (%)', 'Muscle Mass (MM)', 'Total Fat Content (TFC)', 'Visceral Fat Area (VFA)']
Rows remaining as candidates after top-10 filter: 0 (of 319)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,Body Mass Index (BMI),Total Body Water (TBW),Extracellular Water (ECW),Intracellular Water (ICW),Extracellular Fluid/Total Body Water (ECF/TBW),Total Body Fat Ratio (TBFR) (%),Lean Mass (LM) (%),Body Protein Content (Protein) (%),Visceral Fat Rating (VFR),Bone Mass (BM),Muscle Mass (MM),Obesity (%),Total Fat Content (TFC),Visceral Fat Area (VFA),Visceral Muscle Area (VMA) (Kg),Hepatic Fat Accumulation (HFA),Glucose,Total Cholesterol (TC),Low Density Lipoprotein (LDL),High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
0,0,44,0,0,0,0,0,0,179,81.9,25.6,47.3,19.3,28.0,41.0,18.70,81.32,17.63,7,3.3,63.3,16.20,15.3,9.5,34.7,0,91.0,233.0,160.0,54.0,100.0,17.0,20.0,66.0,1.03,91.83,0.63,15.5,10.900000
1,1,42,0,0,0,0,0,0,182,88.0,26.6,47.6,19.0,29.0,40.0,22.05,72.26,18.17,8,2.1,65.2,13.05,19.4,11.5,35.0,1,96.0,336.0,177.0,60.0,159.0,12.0,27.0,65.0,1.01,95.80,0.50,14.9,23.600000
2,0,52,0,1,1,0,0,0,169,88.9,31.1,47.1,19.7,27.4,42.0,26.30,73.68,16.13,13,3.3,62.2,41.60,23.4,14.4,33.7,4,123.0,184.0,100.0,45.0,224.0,25.0,38.0,33.0,1.22,71.33,0.00,14.1,25.142857
3,1,31,0,0,0,0,0,0,178,94.8,29.9,50.9,20.6,30.3,40.0,25.30,74.68,15.54,9,3.5,67.3,36.00,24.0,14.8,35.4,2,97.0,156.0,109.0,34.0,70.0,36.0,24.0,67.0,0.72,125.00,13.90,15.4,13.600000
4,0,38,0,0,0,0,0,0,171,68.6,23.5,39.5,16.6,22.9,42.0,19.20,80.76,17.28,6,2.8,52.6,6.70,13.2,8.2,28.8,0,93.0,239.0,169.0,43.0,129.0,19.0,34.0,75.0,0.91,110.63,0.00,16.6,15.600000


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Gallstone Status,category,0.0,0.0,2.0,"0, 1"
1,Gender,category,0.0,0.0,2.0,"0, 1"
2,Comorbidity,category,0.0,0.0,4.0,"0, 1, 3, 2"
3,Coronary Artery Disease (CAD),category,0.0,0.0,2.0,"0, 1"
4,Hypothyroidism,category,0.0,0.0,2.0,"0, 1"
5,Hyperlipidemia,category,0.0,0.0,2.0,"0, 1"
6,Diabetes Mellitus (DM),category,0.0,0.0,2.0,"0, 1"
7,Weight,float64,0.0,0.0,245.0,"68.4, 63.6, 76.8, 78.4, 69.6, 74.7, 92.2, 68.2, 62.8, 70.6"
8,Body Mass Index (BMI),float64,0.0,0.0,162.0,"28.2, 25.3, 28.0, 29.9, 25.2, 25.1, 27.2, 27.4, 30.8, 26.6"
9,Total Body Water (TBW),float64,0.0,0.0,199.0,"47.1, 34.3, 44.0, 31.2, 36.2, 47.3, 47.6, 31.9, 34.8, 43.2"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,319.0,48.068966,12.114558,20.00,96.00
Height,319.0,167.156740,10.053030,145.00,191.00
Weight,319.0,80.564890,15.709069,42.90,143.50
Body Mass Index (BMI),319.0,28.877116,5.313707,17.40,49.70
Total Body Water (TBW),319.0,40.587774,7.930235,13.00,66.20
Extracellular Water (ECW),319.0,17.071160,3.161857,9.00,27.80
Intracellular Water (ICW),319.0,23.634483,5.349332,13.80,57.10
Extracellular Fluid/Total Body Water (ECF/TBW),319.0,42.212038,3.244470,29.23,52.00
Total Body Fat Ratio (TBFR) (%),319.0,28.274984,8.444417,6.30,50.92
Lean Mass (LM) (%),319.0,71.638245,8.437598,48.99,93.67


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                        rank                    
Comorbidity                   1        0    217  68.03
                              2        1     99  31.03
                              3        3      2   0.63
                              4        2      1   0.31
Coronary Artery Disease (CAD) 1        0    307  96.24
                              2        1     12   3.76
Diabetes Mellitus (DM)        1        0    276  86.52
                              2        1     43  13.48
Gallstone Status              1        0    161  50.47
                              2        1    158  49.53
Gender                        1        0    162  50.78
                              2        1    157  49.22
Hyperlipidemia                1        0    311  97.49
                              2        1      8   2.51
Hypothyroidism                1        0    310  97.18
                              2        1      9   2.82

In [8]:
# Target Distribution
target_df

,count,pct
Gallstone Status,,
0,161,50.47
1,158,49.53


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7c96-2179-77ae-96b3-ca02452ece61
e66bdeb1ec9ba2360a2ffe61292c8bd05d0a2c97695faebf7be419574904ba51
